<a href="https://colab.research.google.com/github/Fawada/AI-ML-course-notebooks/blob/main/module4/m4_l2_vae_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌌 Module 4 · Lesson 2 · Notebook
# Galaxy Morphology Synthesis with a Variational Autoencoder

---

**Module 4 · Generative Models**  
**Estimated time:** 50 minutes  
**Runtime required:** GPU (T4) — enable via Runtime → Change runtime type before starting

---

## 📋 What This Notebook Does

In Module 3 you built a dense autoencoder and VAE on MNIST digits. This notebook scales up to a real scientific task: training a **convolutional Variational Autoencoder** on galaxy images to learn the continuous space of galaxy morphologies and generate new galaxies by sampling from it.

This is not a toy demonstration. Generating synthetic galaxy images has genuine applications — augmenting training sets for morphological classifiers, exploring the distribution of galaxy types in simulated universes, and generating matched samples for statistical comparisons.

By the end of this notebook you will have:

- Built a convolutional VAE with a reparameterisation sampling layer from scratch
- Trained the model while monitoring reconstruction loss and KL loss separately
- Generated new galaxy images by sampling random latent vectors from N(0,1)
- Visualised the latent space with UMAP, coloured by morphological type
- Performed latent space interpolation between two galaxies
- Investigated the effect of the beta hyperparameter on generation quality vs latent structure

---

## 🗺️ Notebook Structure

| Step | What you do |
|------|-------------|
| **0. Setup** | Install packages, imports, check GPU |
| **1. Load Data** | Download and explore the galaxy morphology dataset |
| **2. Preprocess** | Normalise, split, inspect class distribution |
| **3. Build VAE** | Encoder, sampling layer, decoder |
| **4. Train** | Monitor reconstruction + KL loss separately |
| **5. Generate** | Sample from N(0,1) and decode new galaxies |
| **6. Latent Space** | UMAP visualisation coloured by morphology |
| **7. Interpolation** | Transition between two galaxy types |
| **8. Try This!** | Beta experiment + latent arithmetic |
| **9. Reflect** | Guided questions |

---

> **Before you start:** Read the Lesson 2 reading pages (KL divergence, ELBO, reparameterisation, beta-VAE) and the notebook preparation HTML page. The concepts in those pages appear directly in the code below.

---
## Step 0 — Setup

We need `umap-learn` for the latent space visualisation. Everything else is pre-installed on Colab.

In [ ]:
!pip install umap-learn --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import umap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

# Confirm GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
if gpus:
    print(f'✅ GPU available: {gpus[0].name}')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')
    print('   Without GPU, training will be 10-20× slower.')

---
## Step 1 — Load and Explore the Galaxy Dataset

### 1.1 — About the Dataset

We use a simulated dataset based on the structure of real galaxy morphology survey data. Each image is a **64×64 pixel greyscale** galaxy image. The dataset contains three broad morphological types:

| Class | Morphology | Description |
|-------|-----------|-------------|
| 0 | Elliptical | Smooth, featureless, rounded |
| 1 | Spiral | Disc-shaped with arm structure |
| 2 | Irregular | Asymmetric, often merger remnants |

> **Important:** Galaxy morphology is a **continuous spectrum**, not three sharp categories. The class labels are human-assigned boundaries on a fundamentally continuous distribution — which is precisely why a VAE (which learns a continuous latent space) is appropriate here.

In [ ]:
def generate_galaxy(morph_type, n_samples, img_size=64, seed=None):
    """Simulate galaxy images for three morphological types."""
    if seed is not None:
        np.random.seed(seed)
    imgs = []
    for _ in range(n_samples):
        img = np.zeros((img_size, img_size), dtype=np.float32)
        cx, cy = img_size//2 + np.random.randint(-5,6), img_size//2 + np.random.randint(-5,6)

        if morph_type == 0:  # Elliptical — smooth Gaussian blob
            a = np.random.uniform(8, 18)
            b = np.random.uniform(5, a)
            angle = np.random.uniform(0, np.pi)
            brightness = np.random.uniform(0.7, 1.0)
            for x in range(img_size):
                for y in range(img_size):
                    dx, dy = x-cx, y-cy
                    xr =  dx*np.cos(angle) + dy*np.sin(angle)
                    yr = -dx*np.sin(angle) + dy*np.cos(angle)
                    r2 = (xr/a)**2 + (yr/b)**2
                    img[x,y] = brightness * np.exp(-r2 * 2)

        elif morph_type == 1:  # Spiral — disc + arms
            radius = np.random.uniform(12, 22)
            n_arms = np.random.choice([2, 4])
            pitch  = np.random.uniform(0.3, 0.6)
            for x in range(img_size):
                for y in range(img_size):
                    dx, dy = x-cx, y-cy
                    r = np.sqrt(dx**2 + dy**2)
                    theta = np.arctan2(dy, dx)
                    disc  = np.exp(-(r/radius)**2 * 1.5) * 0.6
                    arm_phase = (theta - pitch * np.log(r+1e-6)) * n_arms
                    arms  = np.exp(-(r/radius)**2) * 0.4 * np.cos(arm_phase)**4
                    img[x,y] = np.clip(disc + arms, 0, 1)

        else:  # Irregular — asymmetric clumps
            n_clumps = np.random.randint(2, 5)
            for _ in range(n_clumps):
                ox = cx + np.random.randint(-15, 16)
                oy = cy + np.random.randint(-15, 16)
                sz = np.random.uniform(4, 12)
                brt = np.random.uniform(0.3, 0.9)
                for x in range(img_size):
                    for y in range(img_size):
                        r2 = ((x-ox)/sz)**2 + ((y-oy)/sz)**2
                        img[x,y] = np.clip(img[x,y] + brt*np.exp(-r2*2), 0, 1)

        img += np.random.normal(0, 0.04, img.shape)  # observational noise
        imgs.append(np.clip(img, 0, 1))
    return np.array(imgs)

print('Generating galaxy morphology dataset...')
N_PER_CLASS = 800
X_e = generate_galaxy(0, N_PER_CLASS, seed=42)
X_s = generate_galaxy(1, N_PER_CLASS, seed=43)
X_i = generate_galaxy(2, N_PER_CLASS, seed=44)

X_all = np.concatenate([X_e, X_s, X_i], axis=0)
y_all = np.array([0]*N_PER_CLASS + [1]*N_PER_CLASS + [2]*N_PER_CLASS)
morph_names = ['Elliptical', 'Spiral', 'Irregular']

# Shuffle
idx = np.random.permutation(len(X_all))
X_all, y_all = X_all[idx], y_all[idx]

print(f'Dataset: {X_all.shape[0]} galaxies, {X_all.shape[1]}×{X_all.shape[2]} pixels')
print(f'Classes: {[(n, (y_all==i).sum()) for i, n in enumerate(morph_names)]}')

In [ ]:
# Visualise samples from each morphological class
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
colors = ['#4a90c4', '#2ecc71', '#e67e22']

for row, (morph_idx, morph_name, color) in enumerate(zip(range(3), morph_names, colors)):
    samples = X_all[y_all == morph_idx][:8]
    for col in range(8):
        axes[row, col].imshow(samples[col], cmap='inferno', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(morph_name, fontsize=9, rotation=0,
                                       labelpad=55, va='center', color=color)

plt.suptitle('Galaxy Morphology Dataset — 8 Examples per Class', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Notice the variation within each class — different sizes, orientations, brightness levels.')
print('This within-class variation is what the VAE must learn to represent in its latent space.')

---
## Step 2 — Preprocess

Minimal preprocessing needed — images are already in [0,1]. We add a channel dimension (required by convolutional layers) and split into training and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

IMG_SIZE = 64

# Add channel dimension: (N, 64, 64) → (N, 64, 64, 1)
X = X_all[..., np.newaxis].astype('float32')

X_train, X_test, y_train, y_test = train_test_split(
    X, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print(f'Training:   {X_train.shape}  ({X_train.shape[0]} galaxies)')
print(f'Test:       {X_test.shape}')
print(f'Pixel range: [{X_train.min():.3f}, {X_train.max():.3f}]  ✅')

---
## Step 3 — Build the Convolutional VAE

### 3.1 — Design Decisions

**Encoder:** Three convolutional layers with increasing filter counts (32→64→128), each followed by batch normalisation and stride-2 downsampling. The spatial feature map is then flattened and passed through dense layers to produce `mu` and `log_var`.

**Sampling layer:** Implements the reparameterisation trick as a custom Keras layer — `z = mu + sigma * epsilon` where `epsilon ~ N(0,1)`.

**Decoder:** Mirror of the encoder — dense layer to reshape, then three transposed convolutional layers (deconv) that progressively upsample back to 64×64. Sigmoid activation on the final layer (required for BCE loss).

> **Why `log_var` instead of `var`?** Variance must be positive, but neural network outputs are unbounded. Using `log_var` allows the network to output any real number — we convert back to standard deviation with `exp(0.5 * log_var)`. This is numerically more stable.

In [ ]:
LATENT_DIM = 64   # dimensionality of the latent space
BETA       = 1.0  # KL weight — change this in Step 8 to explore beta-VAE

# ── Sampling layer (reparameterisation trick) ──
class SamplingLayer(keras.layers.Layer):
    """z = mu + sigma * epsilon  (reparameterisation trick)"""
    def call(self, inputs):
        mu, log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(mu))
        sigma   = tf.exp(0.5 * log_var)
        return mu + sigma * epsilon

# ── Encoder ──
def build_encoder(latent_dim):
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name='encoder_input')
    x = layers.Conv2D(32,  3, strides=2, padding='same', name='conv1')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(64,  3, strides=2, padding='same', name='conv2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(128, 3, strides=2, padding='same', name='conv3')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu', name='enc_dense')(x)

    mu      = layers.Dense(latent_dim, name='mu')(x)
    log_var = layers.Dense(latent_dim, name='log_var')(x)
    z       = SamplingLayer(name='z')([mu, log_var])

    return keras.Model(inp, [mu, log_var, z], name='encoder')

# ── Decoder ──
def build_decoder(latent_dim):
    inp     = keras.Input(shape=(latent_dim,), name='decoder_input')
    spatial = IMG_SIZE // 8   # after 3 stride-2 convolutions
    x = layers.Dense(spatial * spatial * 128, activation='relu', name='dec_dense')(inp)
    x = layers.Reshape((spatial, spatial, 128))(x)
    x = layers.Conv2DTranspose(64,  3, strides=2, padding='same', name='deconv1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2DTranspose(32,  3, strides=2, padding='same', name='deconv2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2DTranspose(1,   3, strides=2, padding='same',
                                activation='sigmoid', name='deconv3')(x)
    return keras.Model(inp, x, name='decoder')

encoder = build_encoder(LATENT_DIM)
decoder = build_decoder(LATENT_DIM)

print(f'LATENT_DIM = {LATENT_DIM}  |  BETA = {BETA}')
print()
encoder.summary()
print()
decoder.summary()

### 3.2 — Custom VAE Model with Separate Loss Tracking

We wrap the encoder and decoder in a custom Keras model that tracks reconstruction loss and KL loss separately. This is essential for monitoring training — you need to see both curves to detect KL collapse.

In [ ]:
class GalaxyVAE(keras.Model):
    def __init__(self, encoder, decoder, beta=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta    = beta
        # Separate metrics for monitoring
        self.total_loss_tracker   = keras.metrics.Mean(name='total_loss')
        self.recon_loss_tracker   = keras.metrics.Mean(name='recon_loss')
        self.kl_loss_tracker      = keras.metrics.Mean(name='kl_loss')

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            mu, log_var, z = self.encoder(x, training=True)
            x_recon = self.decoder(z, training=True)

            # Reconstruction loss — BCE per pixel, summed over pixels
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(x, x_recon),
                    axis=(1, 2)
                )
            )
            # KL divergence — closed-form for Gaussian
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + log_var - tf.square(mu) - tf.exp(log_var), axis=1)
            )
            total_loss = recon_loss + self.beta * kl_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {'total': self.total_loss_tracker.result(),
                'recon': self.recon_loss_tracker.result(),
                'kl':    self.kl_loss_tracker.result()}

    def call(self, x):
        mu, log_var, z = self.encoder(x)
        return self.decoder(z)

vae = GalaxyVAE(encoder, decoder, beta=BETA)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))
print(f'✅ VAE compiled — beta = {BETA}')
print(f'   Reconstruction loss and KL loss will be tracked separately during training.')

---
## Step 4 — Train the VAE

Training takes approximately 8–12 minutes on a T4 GPU. Watch both loss curves as they print — the KL loss should start near zero and gradually rise as the encoder learns to spread its distributions. If the KL loss collapses back to zero, re-read the KL collapse section in the notebook preparation page.

In [ ]:
EPOCHS     = 50
BATCH_SIZE = 64

print(f'Training Galaxy VAE for {EPOCHS} epochs...')
print(f'  Batch size: {BATCH_SIZE}  |  Latent dim: {LATENT_DIM}  |  Beta: {BETA}')
print()

history = vae.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print('\n✅ Training complete.')

In [ ]:
# Plot both loss curves separately
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

losses = [
    ('total', 'Total Loss (Recon + β·KL)', 'steelblue'),
    ('recon', 'Reconstruction Loss', '#2ecc71'),
    ('kl',   'KL Divergence Loss',   '#e74c3c')
]

for ax, (key, title, color) in zip(axes, losses):
    ax.plot(history.history[key], color=color, linewidth=2.5)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Loss', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)

plt.suptitle(f'VAE Training Curves  (β = {BETA}, latent_dim = {LATENT_DIM})',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Final reconstruction loss: {history.history["recon"][-1]:.2f}')
print(f'Final KL loss:             {history.history["kl"][-1]:.2f}')
print()
if history.history['kl'][-1] < 1.0:
    print('⚠️  KL loss is very low — possible KL collapse. Check reconstructions carefully.')
    print('   If galaxies look identical regardless of input, try reducing beta or adding KL warmup.')
else:
    print('📊 KL loss is positive — encoder is spreading its distributions beyond the origin.')
    print('   This is the expected behaviour for a well-trained VAE.')

---
## Step 5 — Generate New Galaxies by Sampling

The generative capability of a VAE: sample a random vector from N(0,1) and pass it through the decoder. The encoder is not used at all — only the decoder transforms the random latent code into an image.

> **This is what distinguishes a VAE from a standard autoencoder.** Because the KL regularisation has filled the latent space continuously, any point sampled from N(0,1) should decode into a plausible galaxy image. A standard autoencoder has gaps in its latent space where the decoder produces incoherent outputs.

In [ ]:
N_GENERATE = 16

# Sample from the prior — this is generation
z_samples = np.random.normal(size=(N_GENERATE, LATENT_DIM)).astype('float32')
generated  = decoder.predict(z_samples, verbose=0)

fig, axes = plt.subplots(2, N_GENERATE//2, figsize=(16, 4.5))
axes = axes.flat

for i, ax in enumerate(axes):
    ax.imshow(generated[i, :, :, 0], cmap='inferno', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(f'Sample {i+1}', fontsize=7)

plt.suptitle('Generated Galaxies — Sampled from N(0,1) and Decoded\n'
             'No real galaxy was used as input — these are entirely new synthetic examples',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('📊 Observe the diversity of morphologies in the generated samples.')
print('   Which morphological types appear most frequently?')
print('   Does the distribution reflect the class balance in the training set?')

In [ ]:
# Compare generated to real reconstructions side by side
n_compare = 8
recon = vae.predict(X_test[:n_compare], verbose=0)

fig, axes = plt.subplots(3, n_compare, figsize=(16, 6.5))
row_labels = ['Original', 'Reconstructed', 'Generated (new)']
row_data   = [X_test[:n_compare, :, :, 0],
              recon[:n_compare, :, :, 0],
              generated[:n_compare, :, :, 0]]

for row, (label, data) in enumerate(zip(row_labels, row_data)):
    for col in range(n_compare):
        axes[row, col].imshow(data[col], cmap='inferno', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(label, fontsize=8, rotation=0,
                                       labelpad=75, va='center')

plt.suptitle('Original · Reconstructed · Generated (new) — Side by Side',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Step 6 — Latent Space Visualisation with UMAP

### 6.1 — Why UMAP?

The latent space has 64 dimensions — far too many to plot directly. UMAP (Uniform Manifold Approximation and Projection) reduces the high-dimensional latent codes into a 2D representation while approximately preserving the local neighbourhood structure. Points that are close in 64D latent space should be close in the 2D UMAP projection.

Unlike Module 3 where we used 2D latent codes directly, here the 2D plot is a projection of a higher-dimensional space — treat it as an approximate map, not an exact representation.

In [ ]:
# Extract latent codes (mu vectors) for test set
mu_test, log_var_test, z_test = encoder.predict(X_test, verbose=0)

print(f'Latent codes extracted: {mu_test.shape}  (mu vectors for {len(X_test)} test galaxies)')
print('Running UMAP projection to 2D...')

reducer     = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
umap_coords = reducer.fit_transform(mu_test)

print(f'UMAP complete. Projection shape: {umap_coords.shape}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

morph_colors = ['#4a90c4', '#2ecc71', '#e67e22']

# Left: coloured by morphological class
for morph_idx, (name, color) in enumerate(zip(morph_names, morph_colors)):
    mask = y_test == morph_idx
    ax1.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                c=color, label=name, alpha=0.5, s=12, edgecolors='none')

ax1.set_xlabel('UMAP Component 1', fontsize=11)
ax1.set_ylabel('UMAP Component 2', fontsize=11)
ax1.set_title('Latent Space — Coloured by Morphological Class\n'
              '(2D UMAP projection of 64D latent space)',
              fontsize=11, fontweight='bold')
ax1.legend(title='Morphology', fontsize=9, markerscale=2.5)
ax1.grid(alpha=0.2)

# Right: coloured by reconstruction error
recon_test = vae.predict(X_test, verbose=0)
recon_err  = np.mean((X_test - recon_test)**2, axis=(1,2,3))

sc = ax2.scatter(umap_coords[:,0], umap_coords[:,1],
                 c=recon_err, cmap='RdYlGn_r', s=12, alpha=0.6,
                 vmin=recon_err.min(), vmax=np.percentile(recon_err,95))
plt.colorbar(sc, ax=ax2, label='Reconstruction MSE')
ax2.set_xlabel('UMAP Component 1', fontsize=11)
ax2.set_ylabel('UMAP Component 2', fontsize=11)
ax2.set_title('Latent Space — Coloured by Reconstruction Error\n'
              'Green = well reconstructed, Red = poorly reconstructed',
              fontsize=11, fontweight='bold')
ax2.grid(alpha=0.2)

plt.suptitle('Latent Space Visualisation — 64D VAE Latent Codes Projected to 2D with UMAP',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('📊 What to notice in the left plot:')
print('   — Are the three morphological classes separated, or do they overlap substantially?')
print('   — Unlike MNIST, galaxy morphology is continuous — expect gradients, not sharp clusters.')
print('\n📊 What to notice in the right plot:')
print('   — Do high-reconstruction-error galaxies (red) cluster in a specific region?')
print('   — If so, what morphological type occupies that region?')

---
## Step 7 — Latent Space Interpolation

We select one elliptical and one spiral galaxy from the test set, encode both to their latent codes (mu vectors), and decode a sequence of equally-spaced points along the straight line between them. A well-trained VAE should produce a smooth, visually coherent morphological transition.

In [ ]:
N_STEPS = 12

# Select one galaxy of each type
elliptical_idx = np.where(y_test == 0)[0][0]
spiral_idx     = np.where(y_test == 1)[0][0]

# Encode to latent codes (mu vectors — deterministic representation)
z1, _, _ = encoder.predict(X_test[elliptical_idx:elliptical_idx+1], verbose=0)
z2, _, _ = encoder.predict(X_test[spiral_idx:spiral_idx+1],     verbose=0)

print(f'Elliptical galaxy latent code: shape {z1.shape}')
print(f'Spiral galaxy latent code:     shape {z2.shape}')
print(f'Euclidean distance in latent space: {np.linalg.norm(z1 - z2):.3f}')

# Interpolate
alphas         = np.linspace(0, 1, N_STEPS)
interp_latents = np.array([(1-a)*z1[0] + a*z2[0] for a in alphas], dtype='float32')
interp_images  = decoder.predict(interp_latents, verbose=0)

# Display the sequence
fig, axes = plt.subplots(1, N_STEPS + 2, figsize=(19, 2.5))

axes[0].imshow(X_test[elliptical_idx, :, :, 0], cmap='inferno', vmin=0, vmax=1)
axes[0].set_title('Original\nElliptical', fontsize=7)
axes[0].axis('off')

for i in range(N_STEPS):
    axes[i+1].imshow(interp_images[i, :, :, 0], cmap='inferno', vmin=0, vmax=1)
    axes[i+1].set_title(f'α={alphas[i]:.2f}', fontsize=6)
    axes[i+1].axis('off')

axes[-1].imshow(X_test[spiral_idx, :, :, 0], cmap='inferno', vmin=0, vmax=1)
axes[-1].set_title('Original\nSpiral', fontsize=7)
axes[-1].axis('off')

plt.suptitle('Latent Space Interpolation: Elliptical → Spiral\n'
             'Each frame decoded from a point along the straight line between the two latent codes',
             fontsize=9, y=1.15)
plt.tight_layout()
plt.show()

print('📊 Smooth, coherent transitions = well-organised latent space.')
print('   At what point in the sequence does the morphology appear to change?')
print('   Do brightness and size transition smoothly? What about structural features?')

---
## Step 8 — Try This! Optional Experiments

### Experiment 1 — Effect of Beta

Retrain the VAE with a different beta value and compare:
- **beta = 0.1** — reconstruction dominates, latent space less structured
- **beta = 4.0** — KL dominates, latent space smoother but reconstructions blurrier

Run the cell below to train a comparison model, then re-run Steps 5–7 with it.

In [ ]:
BETA_EXPERIMENT = 4.0   # try: 0.1, 0.5, 2.0, 4.0

enc_exp = build_encoder(LATENT_DIM)
dec_exp = build_decoder(LATENT_DIM)
vae_exp = GalaxyVAE(enc_exp, dec_exp, beta=BETA_EXPERIMENT)
vae_exp.compile(optimizer=keras.optimizers.Adam(1e-3))

print(f'Training comparison VAE with beta = {BETA_EXPERIMENT}...')
hist_exp = vae_exp.fit(X_train, X_train, epochs=EPOCHS,
                       batch_size=BATCH_SIZE, shuffle=True, verbose=0)
print('✅ Done.')

# Compare training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, key, ylabel in zip(axes,
                            ['recon', 'kl'],
                            ['Reconstruction Loss', 'KL Loss']):
    ax.plot(history.history[key], label=f'β = {BETA}',            color='steelblue', linewidth=2)
    ax.plot(hist_exp.history[key], label=f'β = {BETA_EXPERIMENT}', color='tomato',    linewidth=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(ylabel, fontsize=11, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.suptitle(f'Beta Comparison: β={BETA} vs β={BETA_EXPERIMENT}',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Visual comparison
recon_orig = vae.predict(X_test[:8], verbose=0)
recon_exp  = vae_exp.predict(X_test[:8], verbose=0)

fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for i in range(8):
    axes[0,i].imshow(X_test[i,:,:,0], cmap='inferno'); axes[0,i].axis('off')
    axes[1,i].imshow(recon_orig[i,:,:,0], cmap='inferno'); axes[1,i].axis('off')
    axes[2,i].imshow(recon_exp[i,:,:,0],  cmap='inferno'); axes[2,i].axis('off')
axes[0,0].set_ylabel('Original', fontsize=8, rotation=0, labelpad=55, va='center')
axes[1,0].set_ylabel(f'β = {BETA}', fontsize=8, rotation=0, labelpad=55, va='center')
axes[2,0].set_ylabel(f'β = {BETA_EXPERIMENT}', fontsize=8, rotation=0, labelpad=55, va='center')
plt.suptitle(f'Reconstruction quality: β={BETA} vs β={BETA_EXPERIMENT}',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print(f'📊 Higher beta ({BETA_EXPERIMENT}) should produce blurrier reconstructions')
print(f'   but a smoother, more navigable latent space.')
print(f'   Re-run Steps 6 and 7 using enc_exp and dec_exp to compare latent structures.')

### Experiment 2 — Latent Arithmetic

Does the VAE latent space support arithmetic operations? If the latent dimensions independently encode separable physical properties, then vector arithmetic should produce meaningful results.

In [ ]:
# Compute mean latent codes for each class
mean_latents = {}
for morph_idx, name in enumerate(morph_names):
    mask = y_test == morph_idx
    mus, _, _ = encoder.predict(X_test[mask], verbose=0)
    mean_latents[name] = mus.mean(axis=0)
    print(f'{name}: mean latent code computed from {mask.sum()} test galaxies')

# Arithmetic: Spiral - Elliptical + Irregular = ?
z_arithmetic = (mean_latents['Spiral']
                - mean_latents['Elliptical']
                + mean_latents['Irregular'])

# Decode original means and the arithmetic result
to_decode = np.stack([
    mean_latents['Spiral'],
    mean_latents['Elliptical'],
    mean_latents['Irregular'],
    z_arithmetic
], axis=0).astype('float32')

decoded = decoder.predict(to_decode, verbose=0)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
labels = ['Mean\nSpiral', '–   Mean\nElliptical', '+   Mean\nIrregular', '=   Result', '', '']
for i, (ax, label) in enumerate(zip(axes[:4], labels[:4])):
    ax.imshow(decoded[i, :, :, 0], cmap='inferno', vmin=0, vmax=1)
    ax.set_title(label, fontsize=9)
    ax.axis('off')

# Compare to actual Irregular mean
actual_irr = decoder.predict(mean_latents['Irregular'][np.newaxis], verbose=0)
axes[4].imshow(actual_irr[0, :, :, 0], cmap='inferno', vmin=0, vmax=1)
axes[4].set_title('Actual Mean\nIrregular', fontsize=9)
axes[4].axis('off')
axes[5].axis('off')

plt.suptitle('Latent Arithmetic: Spiral − Elliptical + Irregular = ?', fontsize=11, y=1.05)
plt.tight_layout()
plt.show()

print('📊 Does the arithmetic result resemble the Irregular class or something else?')
print('   Successful latent arithmetic is evidence of a disentangled, compositional latent space.')
print('   It tends to work better with higher beta values.')

---
## Step 9 — Guided Reflection

Answer each question by replacing the placeholder text. These are thinking prompts, not right-or-wrong questions.

---

### ❓ Question 1 — KL Loss Behaviour
**Look at your KL loss curve from Step 4. Did it rise gradually and stabilise, or did it collapse? If you saw collapse, what might have caused it? If you did not, what does the stable positive KL loss tell you about the latent space?**

> **Your answer:** *(Replace this text)*

---

### ❓ Question 2 — Latent Space Structure
**Look at the UMAP scatter plot from Step 6. Is galaxy morphology organised as sharp, distinct clusters (like MNIST digits) or as smooth gradients? What does this tell you about the nature of galaxy morphological variation compared to digit class variation?**

> **Your answer:** *(Replace this text)*

---

### ❓ Question 3 — Interpolation Quality
**From Step 7: at what point in the elliptical→spiral interpolation sequence does the morphology visibly change? Do brightness and ellipticity transition more smoothly than structural features like spiral arms? What does this suggest about how the VAE has organised the latent space?**

> **Your answer:** *(Replace this text)*

---

### ❓ Question 4 — Scientific Value
**The reading described data augmentation through synthesis as a key application of generative models in science. Based on what you have seen, what would be the benefits and risks of using VAE-generated galaxy images to augment a training set for a morphological classifier? What would you need to validate before using synthetic galaxies as scientific data?**

> **Your answer:** *(Replace this text)*

---
## ✅ Notebook Complete

| ✅ | Achievement |
|----|-------------|
| ✅ | Built a convolutional VAE with reparameterisation sampling layer from scratch |
| ✅ | Trained while monitoring reconstruction and KL loss curves separately |
| ✅ | Generated new galaxy images by sampling from N(0,1) — encoder not used |
| ✅ | Visualised the 64D latent space with UMAP, coloured by morphological type and reconstruction error |
| ✅ | Performed latent space interpolation between an elliptical and a spiral galaxy |
| ✅ | Investigated the effect of beta on reconstruction quality vs latent space structure |
| ✅ | Explored latent arithmetic using mean class codes |

---

### ➡️ What Comes Next — Lesson 3: GANs

Lesson 3 introduces Generative Adversarial Networks — a completely different approach to generation. Where the VAE uses a principled probabilistic loss function, a GAN uses adversarial competition between two networks. The outputs are typically sharper and more photorealistic, but training is significantly less stable. Lesson 3 covers the minimax training game, mode collapse, and the Wasserstein distance — and applies GANs to super-resolution of telescope imagery.